# 2-2 정형 피처 베이스라인 모델

`is_low_rating_surge` 예측 — 리뷰 텍스트 없이 정형 피처만 사용한 기준 모델.

## 1. 데이터 로드

In [71]:
import pandas as pd
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import roc_auc_score, average_precision_score, classification_report

df = pd.read_parquet("../../data/processed/product_month_labeled.parquet")

## 2. 신뢰도 피처 추가 (Wilson score interval)

In [72]:
from eda import add_reliability_features

df = add_reliability_features(df)  # reliability_ci_width 컬럼 추가

## 3. 최종 피처셋 확정

VIF/상관관계 검토 결과에 따라 6개만 사용 (`avg_rating`, `low_rating_count`, `text_available_ratio`, `past_3m_review_count`, `past_3m_low_rating_count`는 중복이라 제외).

In [73]:
FEATURE_COLUMNS = [
    "review_count",
    "low_rating_ratio",
    "past_3m_low_rating_ratio",
    "mean_helpful_vote",
    "verified_purchase_ratio",
    "reliability_ci_width",
]
TARGET = "is_low_rating_surge"

## 4. 시간순 학습/검증/테스트 분할

랜덤 split 금지 — 미래 달 데이터가 섞이면 데이터 누수 발생.

In [74]:
train_df = df[df["year_month"] < "2022-01"]
valid_df = df[(df["year_month"] >= "2022-01") & (df["year_month"] < "2022-09")]
test_df  = df[df["year_month"] >= "2022-09"]

X_train, y_train = train_df[FEATURE_COLUMNS], train_df[TARGET]
X_valid, y_valid = valid_df[FEATURE_COLUMNS], valid_df[TARGET]
X_test,  y_test  = test_df[FEATURE_COLUMNS],  test_df[TARGET]

print(f"train {len(train_df)} / valid {len(valid_df)} / test {len(test_df)}")
print(f"label ratio  train {y_train.mean():.2%} / valid {y_valid.mean():.2%} / test {y_test.mean():.2%}")

train 54813 / valid 7228 / test 6046
label ratio  train 8.76% / valid 12.92% / test 11.79%


### 4-1. 피처 스케일링

fit은 X_train으로만 — valid/test는 transform만 (누수 방지).

In [75]:
from sklearn.preprocessing import StandardScaler

scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_valid_scaled = scaler.transform(X_valid)
X_test_scaled  = scaler.transform(X_test)

## 5. 결측치 확인

In [76]:
X_train.isna().sum()
X_valid.isna().sum()
X_test.isna().sum()

review_count                0
low_rating_ratio            0
past_3m_low_rating_ratio    0
mean_helpful_vote           0
verified_purchase_ratio     0
reliability_ci_width        0
dtype: int64

## 6. 베이스라인 모델 학습 (로지스틱회귀)

`class_weight="balanced"`로 클래스 불균형(양성 9.47%) 보정.

In [77]:
model = LogisticRegression(class_weight="balanced", max_iter=1000)
model.fit(X_train_scaled, y_train)

,"class_weight class_weight: dict or 'balanced', default=NoneWeights associated with classes in the form ``{class_label: weight}``.If not given, all classes are supposed to have weight one.The ""balanced"" mode uses the values of y to automatically adjustweights inversely proportional to class frequencies in the input dataas ``n_samples / (n_classes * np.bincount(y))``.Note that these weights will be multiplied with sample_weight (passedthrough the fit method) if sample_weight is specified... versionadded:: 0.17 *class_weight='balanced'*",'balanced'
,"max_iter max_iter: int, default=100Maximum number of iterations taken for the solvers to converge.",1000
,"penalty penalty: {'l1', 'l2', 'elasticnet', None}, default='l2'Specify the norm of the penalty:- `None`: no penalty is added;- `'l2'`: add an L2 penalty term and it is the default choice;- `'l1'`: add an L1 penalty term;- `'elasticnet'`: both L1 and L2 penalty terms are added... warning:: Some penalties may not work with some solvers. See the parameter `solver` below, to know the compatibility between the penalty and solver... versionadded:: 0.19 l1 penalty with SAGA solver (allowing 'multinomial' + L1).. deprecated:: 1.8 `penalty` was deprecated in version 1.8 and will be removed in 1.10. Use `l1_ratio` and `C` instead. `l1_ratio=0` for `penalty='l2'`, `l1_ratio=1` for `penalty='l1'`, `l1_ratio` set to any float between 0 and 1 for `penalty='elasticnet'`, and `C=np.inf` for `penalty=None`.",'deprecated'
,"C C: float, default=1.0Inverse of regularization strength; must be a positive float.Like in support vector machines, smaller values specify strongerregularization. `C=np.inf` results in unpenalized logistic regression.For a visual example on the effect of tuning the `C` parameterwith an L1 penalty, see::ref:`sphx_glr_auto_examples_linear_model_plot_logistic_path.py`.",1.0
,"l1_ratio l1_ratio: float, default=0.0The Elastic-Net mixing parameter, with `0 <= l1_ratio <= 1`. Setting`l1_ratio=1` gives a pure L1-penalty, setting `l1_ratio=0` a pure L2-penalty.Any value between 0 and 1 gives an Elastic-Net penalty of the form`l1_ratio * L1 + (1 - l1_ratio) * L2`... warning:: Certain values of `l1_ratio`, i.e. some penalties, may not work with some solvers. See the parameter `solver` below, to know the compatibility between the penalty and solver... versionchanged:: 1.8 Default value changed from None to 0.0... deprecated:: 1.8 `None` is deprecated and will be removed in version 1.10. Always use `l1_ratio` to specify the penalty type.",0.0
,"dual dual: bool, default=FalseDual (constrained) or primal (regularized, see also:ref:`this equation <regularized-logistic-loss>`) formulation. Dual formulationis only implemented for l2 penalty with liblinear solver. Prefer `dual=False`when n_samples > n_features.",False
,"tol tol: float, default=1e-4Tolerance for stopping criteria.",0.0001
,"fit_intercept fit_intercept: bool, default=TrueSpecifies if a constant (a.k.a. bias or intercept) should beadded to the decision function.",True
,"intercept_scaling intercept_scaling: float, default=1Useful only when the solver `liblinear` is usedand `self.fit_intercept` is set to `True`. In this case, `x` becomes`[x, self.intercept_scaling]`,i.e. a ""synthetic"" feature with constant value equal to`intercept_scaling` is appended to the instance vector.The intercept becomes``intercept_scaling * synthetic_feature_weight``... note:: The synthetic feature weight is subject to L1 or L2 regularization as all other features. To lessen the effect of regularization on synthetic feature weight (and therefore on the intercept) `intercept_scaling` has to be increased.",1
,"random_state random_state: int, RandomState instance, default=NoneUsed when ``solver`` == 'sag', 'saga' or 'liblinear' to shuffle thedata. See :term:`Glossary <random_state>` for details.",None
,"solver solver: {'lbfgs', 'liblinear', 'newton-cg', 'newton-cholesky', 'sag', 'saga'}, default='lbfgs'Algorithm to use in the optimization problem. Defaul

## 7. 검증셋 평가

accuracy는 불균형 데이터에 부적절 — PR-AUC / ROC-AUC 사용.

In [78]:
valid_proba = model.predict_proba(X_valid_scaled)[:, 1]

print(f"ROC-AUC: {roc_auc_score(y_valid, valid_proba):.3f}")
print(f"PR-AUC : {average_precision_score(y_valid, valid_proba):.3f}")
print(classification_report(y_valid, model.predict(X_valid_scaled)))

ROC-AUC: 0.601
PR-AUC : 0.154
              precision    recall  f1-score   support

           0       0.91      0.50      0.65      6294
           1       0.17      0.68      0.27       934

    accuracy                           0.52      7228
   macro avg       0.54      0.59      0.46      7228
weighted avg       0.82      0.52      0.60      7228



## 8. 피처 중요도 (계수 해석)

In [79]:
coef_df = pd.DataFrame({
    "feature": FEATURE_COLUMNS,
    "coefficient": model.coef_[0],
}).sort_values("coefficient", ascending=False)

coef_df

,feature,coefficient
2,past_3m_low_rating_ratio,0.314557
5,reliability_ci_width,0.310971
3,mean_helpful_vote,0.018780
1,low_rating_ratio,0.004732
4,verified_purchase_ratio,-0.030893
0,review_count,-0.274591


## 9. (비교군) LightGBM

다중공선성에 덜 민감한 트리 모델과 성능 비교.

In [80]:
import lightgbm as lgb

lgb_model = lgb.LGBMClassifier(
    scale_pos_weight=(y_train == 0).sum() / (y_train == 1).sum()
)
lgb_model.fit(X_train, y_train)

lgb_proba = lgb_model.predict_proba(X_valid)[:, 1]
print(f"LightGBM ROC-AUC: {roc_auc_score(y_valid, lgb_proba):.3f}")
print(f"LightGBM PR-AUC : {average_precision_score(y_valid, lgb_proba):.3f}")

[LightGBM] [Info] Number of positive: 4801, number of negative: 50012
[LightGBM] [Info] Auto-choosing row-wise multi-threading, the overhead of testing was 0.004503 seconds.
You can set `force_row_wise=true` to remove the overhead.
And if memory is not enough, you can set `force_col_wise=true`.
[LightGBM] [Info] Total Bins 1382
[LightGBM] [Info] Number of data points in the train set: 54813, number of used features: 6
[LightGBM] [Info] [binary:BoostFromScore]: pavg=0.087589 -> initscore=-2.343439
[LightGBM] [Info] Start training from score -2.343439
LightGBM ROC-AUC: 0.624
LightGBM PR-AUC : 0.173


## 10. 최종 테스트셋 평가

검증셋으로 더 좋았던 모델을 골라 마지막에 한 번만 테스트셋에 적용.

In [81]:
test_proba = model.predict_proba(X_test_scaled)[:, 1]

print(f"Test ROC-AUC: {roc_auc_score(y_test, test_proba):.3f}")
print(f"Test PR-AUC : {average_precision_score(y_test, test_proba):.3f}")

Test ROC-AUC: 0.616
Test PR-AUC : 0.145


## 11. Walk-forward 반복검증

최종 test(2022-09~)는 재사용하지 않고, 그 이전 구간만 여러 번 잘라서 검증.

In [82]:
def evaluate_fold(train_df, valid_df):
    X_tr, y_tr = train_df[FEATURE_COLUMNS], train_df[TARGET]
    X_va, y_va = valid_df[FEATURE_COLUMNS], valid_df[TARGET]

    scaler = StandardScaler()
    X_tr_s = scaler.fit_transform(X_tr)
    X_va_s = scaler.transform(X_va)

    model = LogisticRegression(class_weight="balanced", max_iter=1000)
    model.fit(X_tr_s, y_tr)
    proba = model.predict_proba(X_va_s)[:, 1]

    return {
        "n_train": len(train_df),
        "n_valid": len(valid_df),
        "valid_label_ratio": y_va.mean(),
        "roc_auc": roc_auc_score(y_va, proba),
        "pr_auc": average_precision_score(y_va, proba),
    }

cutoff = df[df["year_month"] < "2022-09"]  # 최종 test 제외
months = sorted(cutoff["year_month"].unique())

MIN_TRAIN_MONTHS = 24  # 최소 2년 학습 후 검증 시작
VALID_WINDOW = 6       # 검증 구간 6개월씩 이동

results = []
start = MIN_TRAIN_MONTHS
while start + VALID_WINDOW <= len(months):
    train_end = months[start]
    valid_months = months[start:start + VALID_WINDOW]

    train_fold = cutoff[cutoff["year_month"] < train_end]
    valid_fold = cutoff[cutoff["year_month"].isin(valid_months)]

    m = evaluate_fold(train_fold, valid_fold)
    m["valid_start"], m["valid_end"] = valid_months[0], valid_months[-1]
    results.append(m)
    start += VALID_WINDOW

results_df = pd.DataFrame(results)
results_df

,n_train,n_valid,valid_label_ratio,roc_auc,pr_auc,valid_start,valid_end
0,12587,3775,0.079205,0.683861,0.123055,2018-01,2018-06
1,16362,3634,0.078976,0.694682,0.133643,2018-07,2018-12
2,19996,4612,0.070035,0.672820,0.110615,2019-01,2019-06
3,24608,6085,0.075596,0.659255,0.112183,2019-07,2019-12
4,30693,5352,0.099028,0.650538,0.140813,2020-01,2020-06
5,36045,6149,0.092698,0.643051,0.126794,2020-07,2020-12
6,42194,7133,0.108930,0.657972,0.160405,2021-01,2021-06
7,49327,5486,0.121764,0.624436,0.155518,2021-07,2021-12
8,54813,5203,0.132616,0.599943,0.158461,2022-01,2022-06


In [83]:
# 요약: 구간별 성능이 얼마나 안정적인지
results_df[["roc_auc", "pr_auc"]].agg(["mean", "std"])

,roc_auc,pr_auc
mean,0.654062,0.135721
std,0.029309,0.019278


In [84]:
def fit_and_eval(feature_cols):
    X_tr = train_df[feature_cols]
    X_va = valid_df[feature_cols]

    scaler = StandardScaler()
    X_tr_s = scaler.fit_transform(X_tr)
    X_va_s = scaler.transform(X_va)

    m = LogisticRegression(class_weight="balanced", max_iter=1000)
    m.fit(X_tr_s, y_train)
    proba = m.predict_proba(X_va_s)[:, 1]

    print(f"  ROC-AUC: {roc_auc_score(y_valid, proba):.3f} / PR-AUC: {average_precision_score(y_valid, proba):.3f}")
    for col, coef in sorted(zip(feature_cols, m.coef_[0]), key=lambda x: -x[1]):
        print(f"    {col:30s} {coef:.4f}")

print("[A] past_3m_low_rating_ratio 제외 (low_rating_ratio만 남김)")
fit_and_eval([c for c in FEATURE_COLUMNS if c != "past_3m_low_rating_ratio"])

print("\n[B] low_rating_ratio 제외 (past_3m_low_rating_ratio만 남김)")
fit_and_eval([c for c in FEATURE_COLUMNS if c != "low_rating_ratio"])

print("\n[C] 원래 6개 다 포함 (비교 기준)")
fit_and_eval(FEATURE_COLUMNS)

[A] past_3m_low_rating_ratio 제외 (low_rating_ratio만 남김)
  ROC-AUC: 0.591 / PR-AUC: 0.153
    reliability_ci_width           0.3274
    low_rating_ratio               0.2156
    mean_helpful_vote              0.0163
    verified_purchase_ratio        -0.0306
    review_count                   -0.2559

[B] low_rating_ratio 제외 (past_3m_low_rating_ratio만 남김)
  ROC-AUC: 0.601 / PR-AUC: 0.154
    past_3m_low_rating_ratio       0.3178
    reliability_ci_width           0.3126
    mean_helpful_vote              0.0190
    verified_purchase_ratio        -0.0309
    review_count                   -0.2729

[C] 원래 6개 다 포함 (비교 기준)
  ROC-AUC: 0.601 / PR-AUC: 0.154
    past_3m_low_rating_ratio       0.3146
    reliability_ci_width           0.3110
    mean_helpful_vote              0.0188
    low_rating_ratio               0.0047
    verified_purchase_ratio        -0.0309
    review_count                   -0.2746
